# Advanced Patterns

Advanced algorithmic patterns that frequently appear in interviews.

## Topics Covered
- Two Pointers (Opposite Direction, Same Direction)
- Meet in the Middle (Subset Sum)
- Sweep Line Algorithm (Merge Intervals, Interval Intersection)
- Fast & Slow Pointers
- Merge Intervals Pattern
- Cyclic Sort

---
## 1. Two Pointers - Opposite Direction

Two pointers start at opposite ends and move toward each other. Common for sorted arrays and palindrome problems.

In [ ]:
def two_sum_sorted(numbers, target):
    """Find two numbers that sum to target in sorted array."""
    left, right = 0, len(numbers) - 1
    while left < right:
        curr = numbers[left] + numbers[right]
        if curr == target:
            return [left + 1, right + 1]
        elif curr < target:
            left += 1
        else:
            right -= 1
    return []

def three_sum(nums):
    """Find all unique triplets that sum to zero."""
    nums.sort()
    result = []
    n = len(nums)
    for i in range(n - 2):
        if i > 0 and nums[i] == nums[i-1]:
            continue  # Skip duplicates
        left, right = i + 1, n - 1
        target = -nums[i]
        while left < right:
            total = nums[left] + nums[right]
            if total == target:
                result.append([nums[i], nums[left], nums[right]])
                while left < right and nums[left] == nums[left+1]:
                    left += 1
                while left < right and nums[right] == nums[right-1]:
                    right -= 1
                left += 1
                right -= 1
            elif total < target:
                left += 1
            else:
                right -= 1
    return result

print(f"Two sum sorted: {two_sum_sorted([2,7,11,15], 9)}")  # [1, 2]
print(f"Three sum: {three_sum([-1,0,1,2,-1,-4])}")  # [[-1,-1,2],[-1,0,1]]
# Time: O(n) for two sum, O(n²) for three sum

---
## 2. Two Pointers - Same Direction

Both pointers start at the same end, one moves faster. Used for in-place removal, sliding window.

In [ ]:
def remove_duplicates(nums):
    """Remove duplicates in-place from sorted array."""
    if not nums:
        return 0
    slow = 0
    for fast in range(1, len(nums)):
        if nums[fast] != nums[slow]:
            slow += 1
            nums[slow] = nums[fast]
    return slow + 1

def move_zeroes(nums):
    """Move all zeroes to end, maintaining relative order."""
    slow = 0
    for fast in range(len(nums)):
        if nums[fast] != 0:
            nums[slow], nums[fast] = nums[fast], nums[slow]
            slow += 1
    return nums

print(f"Remove duplicates: {remove_duplicates([0,0,1,1,1,2,2,3,3,4])}")  # 5
print(f"Move zeroes: {move_zeroes([0,1,0,3,12])}")  # [1,3,12,0,0]
# Time: O(n), Space: O(1)

---
## 3. Meet in the Middle

Split the problem into two halves, solve each, and combine. Reduces 2^n to 2^(n/2). Useful for subset sum.

In [ ]:
def subset_sum_meet_in_middle(nums, target):
    """Check if any subset sums to target. Uses meet-in-the-middle for O(n*2^(n/2))."""
    n = len(nums)
    # Split into two halves
    left_half = nums[:n//2]
    right_half = nums[n//2:]

    def generate_sums(arr):
        """Generate all possible subset sums."""
        sums = []
        for mask in range(1 << len(arr)):
            total = 0
            for i in range(len(arr)):
                if mask & (1 << i):
                    total += arr[i]
            sums.append(total)
        return sums

    left_sums = generate_sums(left_half)
    right_sums = generate_sums(right_half)
    right_sums.sort()

    for s in left_sums:
        # Find if target - s exists in right_sums
        complement = target - s
        import bisect
        i = bisect.bisect_left(right_sums, complement)
        if i < len(right_sums) and right_sums[i] == complement:
            return True
    return False

print(f"Subset sum 9 from [3,5,7,1,9,2]: {subset_sum_meet_in_middle([3,5,7,1,9,2], 9)}")  # True
# Time: O(n * 2^(n/2)), Space: O(2^(n/2))

---
## 4. Sweep Line Algorithm

Process events in order. Used for merge intervals, interval intersection, skyline problem.

In [ ]:
def merge_intervals(intervals):
    """Merge overlapping intervals."""
    if not intervals:
        return []
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:  # Overlap
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

def interval_intersection(A, B):
    """Find intersection of two interval lists."""
    result = []
    i = j = 0
    while i < len(A) and j < len(B):
        start = max(A[i][0], B[j][0])
        end = min(A[i][1], B[j][1])
        if start <= end:
            result.append([start, end])
        if A[i][1] < B[j][1]:
            i += 1
        else:
            j += 1
    return result

intervals = [[1,3],[2,6],[8,10],[15,18]]
A = [[0,2],[5,10],[13,23],[24,25]]
B = [[1,5],[8,12],[15,24],[25,26]]
print(f"Merged: {merge_intervals(intervals)}")  # [[1,6],[8,10],[15,18]]
print(f"Intersection: {interval_intersection(A, B)}")
# Time: O(n log n) for merge (sort), O(n + m) for intersection

---
## 5. Fast & Slow Pointers (Tortoise and Hare)

Used for cycle detection in linked lists and arrays.

In [ ]:
def find_duplicate(nums):
    """Find duplicate number in nums (Floyd's cycle detection)."""
    slow = fast = nums[0]
    while True:
        slow = nums[slow]
        fast = nums[nums[fast]]
        if slow == fast:
            break
    slow = nums[0]
    while slow != fast:
        slow = nums[slow]
        fast = nums[fast]
    return slow

def find_cycle_start(head):
    """Find start of cycle in linked list."""
    slow = fast = head
    has_cycle = False
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:
            has_cycle = True
            break
    if not has_cycle:
        return None
    slow = head
    while slow != fast:
        slow = slow.next
        fast = fast.next
    return slow

print(f"Duplicate in [1,3,4,2,2]: {find_duplicate([1,3,4,2,2])}")  # 2
# Time: O(n), Space: O(1)

---
## 6. Cyclic Sort

Sort numbers in range [1..n] in O(n) time by placing each number at its correct index.

In [ ]:
def cyclic_sort(nums):
    """Sort array of 1..n in place in O(n)."""
    i = 0
    while i < len(nums):
        correct = nums[i] - 1  # Correct index for value nums[i]
        if nums[i] != nums[correct]:
            nums[i], nums[correct] = nums[correct], nums[i]
        else:
            i += 1
    return nums

def find_missing_number(nums):
    """Find missing number in 0..n using cyclic sort."""
    n = len(nums)
    i = 0
    while i < n:
        correct = nums[i]
        if 0 < nums[i] < n and nums[i] != nums[correct]:
            nums[i], nums[correct] = nums[correct], nums[i]
        else:
            i += 1
    for i in range(n):
        if nums[i] != i:
            return i
    return n

print(f"Cyclic sort [3,1,5,4,2]: {cyclic_sort([3,1,5,4,2])}")  # [1,2,3,4,5]
print(f"Missing number: {find_missing_number([3,0,1])}")  # 2
print(f"Missing number: {find_missing_number([0,1])}")    # 2
# Time: O(n), Space: O(1)

---
## 7. Monotonic Queue (Deque-based)

Maintain elements in increasing/decreasing order. Used for sliding window max/min.

In [ ]:
from collections import deque

def max_sliding_window(nums, k):
    """Sliding window maximum using monotonic deque."""
    dq = deque()  # Indices with decreasing values
    result = []
    for i, num in enumerate(nums):
        while dq and dq[0] <= i - k:
            dq.popleft()
        while dq and nums[dq[-1]] <= num:
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            result.append(nums[dq[0]])
    return result

print(f"Sliding max: {max_sliding_window([1,3,-1,-3,5,3,6,7], 3)}")
# Time: O(n), Space: O(k)

---
## LeetCode Problems

### LeetCode 15 - 3Sum (Two Pointers)

**Problem:** Find all unique triplets that sum to zero.

In [ ]:
def threeSum(nums):
    nums.sort()
    result = []
    n = len(nums)
    for i in range(n - 2):
        if i > 0 and nums[i] == nums[i-1]:
            continue
        l, r = i + 1, n - 1
        target = -nums[i]
        while l < r:
            s = nums[l] + nums[r]
            if s == target:
                result.append([nums[i], nums[l], nums[r]])
                while l < r and nums[l] == nums[l+1]: l += 1
                while l < r and nums[r] == nums[r-1]: r -= 1
                l += 1; r -= 1
            elif s < target:
                l += 1
            else:
                r -= 1
    return result
# Time: O(n²), Space: O(1) (excluding output)

### LeetCode 56 - Merge Intervals (Sweep Line)

**Problem:** Merge all overlapping intervals.

In [ ]:
def merge(intervals):
    if not intervals:
        return []
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return merged
# Time: O(n log n), Space: O(n)

---
## Summary

| Pattern | Technique | Time | Space | Use Case |
|---|---|---|---|---|
| Two Pointers (opp) | Left + Right | O(n) | O(1) | Sorted arrays, 3Sum |
| Two Pointers (same) | Slow + Fast | O(n) | O(1) | In-place removal |
| Meet in Middle | Split & combine | O(n*2^(n/2)) | O(2^(n/2)) | Subset sum |
| Sweep Line | Sort + process | O(n log n) | O(n) | Intervals |
| Fast & Slow | Floyd's cycle | O(n) | O(1) | Cycle detection |
| Cyclic Sort | Place at index | O(n) | O(1) | 1..n range numbers |
| Monotonic Queue | Deque | O(n) | O(k) | Sliding window |